# Figures and metrics

This notebook does not train. It reads `artifacts/` and builds the five figures plus the comparison table.

Figure 1 asks whether sparsity and the KL term soften reconstructions. Figure 2 asks which prior sample is sharp and which covers more modes. Figure 3 is the classifier limitation. Figure 4 asks whether a straight line in latent space stays a plausible image. Figure 5 is the training trace, with the diffusion reverse chain underneath.

Run this on a Colab **GPU** runtime. The project folder must sit at `MyDrive/dl-sesi07`. Change `DRIVE_ROOT` in the setup cell if your folder name differs. Run the cells from top to bottom. Results are written to `artifacts/` on Drive. Run this after notebooks 01–07 have each written their `artifacts/<model>/` folder.


In [ ]:
import os
import sys
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/dl-sesi07"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path(DRIVE_ROOT)
else:
    ROOT = Path.cwd().resolve()
    if ROOT.name == "notebooks":
        ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if "google.colab" in sys.modules:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])


In [ ]:
%matplotlib inline
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from matplotlib.gridspec import GridSpec

from src.config import CLASS_NAMES, FID_SAMPLES, get_device
from src.data import get_splits, make_loader
from src.io import load_json, load_shared, load_torch
from src.metrics import fid_score, real_images_for_fid

DEVICE = get_device()
print("device", DEVICE)
FIGURES = Path("artifacts") / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["figure.dpi"] = 120

test8 = load_shared("test8.pt")
originals = test8["images"]

def need(name, filename):
    return load_torch(name, filename)

vanilla_recon = need("vanilla_ae", "recon.pt")
sparse_recon = need("sparse_ae", "recon.pt")
vae_recon = need("vae", "recon.pt")
vae_grid = need("vae", "samples_grid.pt")
gan_grid = need("vanilla_gan", "samples_grid.pt")
dcgan_grid = need("dcgan", "samples_grid.pt")
ddpm_grid = need("ddpm", "samples_grid.pt")
print("loaded figure tensors")


## Figure 1. Reconstruction

Row 1 is the test image. Rows 2–4 are vanilla AE, sparse AE, and VAE. Look for soft edges, and for whether the sparse codes kept or lost local detail relative to the dense autoencoder.


In [ ]:
names = ["Original", "Vanilla AE", "Sparse AE", "VAE"]
rows = [originals, vanilla_recon, sparse_recon, vae_recon]
fig, axes = plt.subplots(4, 8, figsize=(12, 6.4))
for row, (name, images) in enumerate(zip(names, rows)):
    axes[row, 0].set_ylabel(name, rotation=0, ha="right", va="center", labelpad=28)
    for col in range(8):
        axes[row, col].imshow(images[col, 0], cmap="gray", vmin=0, vmax=1)
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
fig.suptitle("Figure 1. Same eight test images, reconstructed")
fig.tight_layout()
fig.savefig(FIGURES / "figure1_reconstruction.png", bbox_inches="tight")
plt.show()

vanilla_metrics = load_json("vanilla_ae", "metrics.json")
sparse_metrics = load_json("sparse_ae", "metrics.json")
print(
    "mean |z|  vanilla",
    round(vanilla_metrics["mean_abs_latent"], 4),
    " sparse",
    round(sparse_metrics["mean_abs_latent"], 4),
)
print(
    "fraction |z|<1e-3  vanilla",
    round(vanilla_metrics["frac_near_zero"], 4),
    " sparse",
    round(sparse_metrics["frac_near_zero"], 4),
)


## Figure 2. Random samples

Each panel is a 4×4 grid from the model's prior. VAE, vanilla GAN, and DCGAN decode `z ~ N(0, I)`. DDPM starts from the same kind of Gaussian image and runs the reverse chain. Compare edge sharpness and whether many samples collapse to one shape.


In [ ]:
panels = [
    ("A  VAE", vae_grid),
    ("B  Vanilla GAN", gan_grid),
    ("C  DCGAN", dcgan_grid),
    ("D  DDPM", ddpm_grid),
]
fig = plt.figure(figsize=(8.6, 9.2))
outer = GridSpec(2, 2, figure=fig, hspace=0.22, wspace=0.06)
for index, (title, images) in enumerate(panels):
    inner = outer[index].subgridspec(4, 4, wspace=0.04, hspace=0.04)
    for cell in range(16):
        axis = fig.add_subplot(inner[cell])
        axis.imshow(images[cell, 0], cmap="gray", vmin=0, vmax=1)
        axis.set_xticks([])
        axis.set_yticks([])
        if cell == 0:
            axis.set_title(title, loc="left")
fig.suptitle("Figure 2. Sixteen samples from each prior")
fig.savefig(FIGURES / "figure2_samples.png", bbox_inches="tight")
plt.show()


## Figure 3. Classifier weights are not a generator

Panel 1 is the real test image. Panel 2 is the transposed-weight projection, first from a class one-hot, then from the image's own features. Panel 3 is a decoder trained on frozen classifier features, next to the vanilla autoencoder at the same latent size. Panel 4 samples the empirical Gaussian of those features. A low classifier error does not imply that any of these paths reconstructs the garment.


In [ ]:
class_proj = need("mlp", "class_projections.pt")
latent_proj = need("mlp", "latent_projections.pt")
frozen = need("mlp", "frozen_recon.pt")
prior = need("mlp", "prior_samples.pt")
mlp_metrics = load_json("mlp", "metrics.json")

fig = plt.figure(figsize=(12, 11))
sections = GridSpec(4, 1, figure=fig, height_ratios=[1.1, 2.2, 2.2, 2.2], hspace=0.45)

top = sections[0].subgridspec(1, 8, wspace=0.04)
for col in range(8):
    axis = fig.add_subplot(top[col])
    axis.imshow(originals[col, 0], cmap="gray", vmin=0, vmax=1)
    axis.axis("off")
    if col == 0:
        axis.set_title("1  Original", loc="left")

middle = sections[1].subgridspec(2, 10, wspace=0.04, hspace=0.15)
for col in range(10):
    axis = fig.add_subplot(middle[0, col])
    axis.imshow(class_proj[col, 0], cmap="gray", vmin=0, vmax=1)
    axis.axis("off")
    if col == 0:
        axis.set_title("2  Class one-hot transposed", loc="left")
for col in range(8):
    axis = fig.add_subplot(middle[1, col])
    axis.imshow(latent_proj[col, 0], cmap="gray", vmin=0, vmax=1)
    axis.axis("off")
    if col == 0:
        axis.set_title("Latent features transposed", loc="left")

compare = sections[2].subgridspec(2, 8, wspace=0.04, hspace=0.2)
for col in range(8):
    axis = fig.add_subplot(compare[0, col])
    axis.imshow(frozen[col, 0], cmap="gray", vmin=0, vmax=1)
    axis.axis("off")
    if col == 0:
        axis.set_title("3  Frozen MLP decoder", loc="left")
    axis = fig.add_subplot(compare[1, col])
    axis.imshow(vanilla_recon[col, 0], cmap="gray", vmin=0, vmax=1)
    axis.axis("off")
    if col == 0:
        axis.set_title("Vanilla AE", loc="left")

samples = sections[3].subgridspec(2, 8, wspace=0.04, hspace=0.15)
for index in range(16):
    axis = fig.add_subplot(samples[index // 8, index % 8])
    axis.imshow(prior[index, 0], cmap="gray", vmin=0, vmax=1)
    axis.axis("off")
    if index == 0:
        axis.set_title("4  Sampled classifier latent", loc="left")

fig.suptitle("Figure 3. MLP classifier weights and features do not generate the image")
fig.savefig(FIGURES / "figure3_mlp.png", bbox_inches="tight")
plt.show()
print(
    "frozen decoder MSE",
    round(mlp_metrics["frozen_mse"], 4),
    " vanilla AE MSE",
    round(vanilla_metrics["test_mse"], 4),
)
print("classifier test accuracy", round(mlp_metrics["test_accuracy"], 4))


## Figure 4. Linear interpolation

`z(alpha) = (1 - alpha) z_A + alpha z_B` for alpha in `{0, 0.2, 0.4, 0.6, 0.8, 1}`. Autoencoders and the VAE encode a T-shirt/top and an ankle boot. The two GANs have no encoder, so they walk between two fixed noise vectors. Broken frames in the middle of an autoencoder row are latent-space voids. DDPM is omitted because it has no code of this form.


In [ ]:
strips = [
    ("Vanilla AE", need("vanilla_ae", "interp.pt")),
    ("Sparse AE", need("sparse_ae", "interp.pt")),
    ("VAE", need("vae", "interp.pt")),
    ("Vanilla GAN", need("vanilla_gan", "interp.pt")),
    ("DCGAN", need("dcgan", "interp.pt")),
]
alphas = ["0.0", "0.2", "0.4", "0.6", "0.8", "1.0"]
fig, axes = plt.subplots(5, 6, figsize=(10, 8))
for row, (name, images) in enumerate(strips):
    axes[row, 0].set_ylabel(name, rotation=0, ha="right", va="center", labelpad=36)
    for col in range(6):
        axes[row, col].imshow(images[col, 0], cmap="gray", vmin=0, vmax=1)
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
        if row == 0:
            axes[row, col].set_title(alphas[col])
fig.suptitle("Figure 4. Linear latent interpolation")
fig.tight_layout()
fig.savefig(FIGURES / "figure4_interpolation.png", bbox_inches="tight")
plt.show()
print("autoencoder endpoints:", CLASS_NAMES[0], "to", CLASS_NAMES[9])


## Figure 5. Loss curves and the diffusion trajectory

Autoencoder curves are mean pixel MSE. GAN panels show the generator and the discriminator separately. The DDPM curve is the noise-prediction MSE. The strip under the curves is one reverse chain from pure noise to the final image.


In [ ]:
vanilla_hist = load_json("vanilla_ae", "history.json")
sparse_hist = load_json("sparse_ae", "history.json")
vae_hist = load_json("vae", "history.json")
gan_hist = load_json("vanilla_gan", "history.json")
dcgan_hist = load_json("dcgan", "history.json")
ddpm_hist = load_json("ddpm", "history.json")
trajectory = need("ddpm", "trajectory.pt")

fig = plt.figure(figsize=(12, 8.5))
grid = GridSpec(3, 2, figure=fig, height_ratios=[1, 1, 0.85], hspace=0.38, wspace=0.28)

axis = fig.add_subplot(grid[0, 0])
axis.plot(vanilla_hist["recon_loss"], label="Vanilla AE")
axis.plot(sparse_hist["recon_loss"], label="Sparse AE")
axis.plot(vae_hist["recon_loss"], label="VAE")
axis.set_title("Reconstruction MSE")
axis.set_xlabel("epoch")
axis.legend(frameon=False)

axis = fig.add_subplot(grid[0, 1])
axis.plot(gan_hist["g_loss"], label="Generator")
axis.plot(gan_hist["d_loss"], label="Discriminator")
axis.set_title("Vanilla GAN")
axis.set_xlabel("epoch")
axis.legend(frameon=False)

axis = fig.add_subplot(grid[1, 0])
axis.plot(dcgan_hist["g_loss"], label="Generator")
axis.plot(dcgan_hist["d_loss"], label="Discriminator")
axis.set_title("DCGAN")
axis.set_xlabel("epoch")
axis.legend(frameon=False)

axis = fig.add_subplot(grid[1, 1])
axis.plot(ddpm_hist["train_loss"], label="Noise MSE")
axis.set_title("DDPM")
axis.set_xlabel("epoch")
axis.legend(frameon=False)

steps = trajectory["steps"]
images = trajectory["images"]
strip = grid[2, :].subgridspec(1, len(steps), wspace=0.05)
for index, step in enumerate(steps):
    axis = fig.add_subplot(strip[index])
    axis.imshow(images[index, 0], cmap="gray", vmin=0, vmax=1)
    axis.set_title(str(step), fontsize=9)
    axis.axis("off")

fig.suptitle("Figure 5. Training loss and one DDPM reverse chain")
fig.savefig(FIGURES / "figure5_curves.png", bbox_inches="tight")
plt.show()


## Metrics table

FID uses the same Inception-v3 pool features for every generative model: grayscale copied to three channels, resized to 299, ImageNet normalization. Real features are cached after the first run. Autoencoders have no FID. GANs and DDPM have no reconstruction score.


In [ ]:
_, _, test_set = get_splits()
real = real_images_for_fid(make_loader(test_set, shuffle=False))
fid_names = {
    "vae": "vae",
    "vanilla_gan": "vanilla_gan",
    "dcgan": "dcgan",
    "ddpm": "ddpm",
}
fid = {}
for name in fid_names:
    fake = load_torch(name, "samples_fid.pt")
    if fake.size(0) < FID_SAMPLES:
        raise RuntimeError(f"{name} saved {fake.size(0)} FID images, expected {FID_SAMPLES}")
    fid[name] = fid_score(fake, real, DEVICE)
    print(name, "FID", round(fid[name], 2))

rows = []
order = [
    ("vanilla_ae", "Vanilla Autoencoder", None),
    ("sparse_ae", "Sparse Autoencoder", None),
    ("vae", "Variational Autoencoder", "vae"),
    ("vanilla_gan", "Vanilla GAN", "vanilla_gan"),
    ("dcgan", "DCGAN", "dcgan"),
    ("ddpm", "DDPM", "ddpm"),
]
for key, label, fid_key in order:
    metrics = load_json(key, "metrics.json")
    rows.append(
        {
            "model": label,
            "latent_dim": metrics["latent_dim"],
            "sec_per_epoch": round(metrics["sec_per_epoch"], 2),
            "latency_ms": round(metrics["latency_ms"], 2),
            "test_mse": "" if metrics["test_mse"] is None else round(metrics["test_mse"], 5),
            "test_psnr": "" if metrics["test_psnr"] is None else round(metrics["test_psnr"], 2),
            "test_ssim": "" if metrics["test_ssim"] is None else round(metrics["test_ssim"], 4),
            "fid": "" if fid_key is None else round(fid[fid_key], 2),
            "stability": metrics["stability"],
        }
    )

fieldnames = list(rows[0].keys())
table_path = Path("artifacts") / "metrics.csv"
with table_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(table_path)
for row in rows:
    print(row)
